# Modeling

In this section, we extend our preprocessing pipeline to train and evaluate classification algorithms. Our goal is to identify the best-performing model for predicting customer churn.

## Imports and splitting data

Firstly, we will use our data preprocessing function and split our data for training and test part. The important thing in splitting will be `stratify=y`. This parameter will ensure that in training and test set there will be the same propotion of each class.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

from utils import split_columns, data_preprocessing

X, y, pre_pipeline = data_preprocessing()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Model Selection and Hyperparameter Tuning

In this experiment, we evaluate five distinct classifiers: **Support Vector Machine (SVM)**, **Decision Tree**, **Random Forest**, **KNN** and **Logistic Regeression**. All models are initialized with `class_weight='balanced'` to account for any imbalances in the target classes.

We utilize **Grid Search** to explore the following hyperparameter spaces:


### 1. K-Nearest Neighbors (KNN)
KNN is a distance-based instance learner that classifies a data point based on how the majority of its "neighbors" are classified. It is highly sensitive to the scale of features, which is why the previous scaling step is critical.
* **`n_neighbors`** `[3, 5, 7, 11, 15]`: The number of nearby points to consider. We use odd numbers to avoid ties in classification voting.
* **`weights`**`['uniform', 'distance']`: Determines the influence of neighbors. Uniform treats all neighbors equally, while Distance gives more weight to closer neighbors, which helps handle class imbalance.
* **`metric`** `['euclidean', 'manhattan']`: The distance formula used. Euclidean is a straight-line distance, while Manhattan (city-block) can be more robust in high-dimensional feature spaces.

### 2. Logistic Regression
A linear model that estimates the probability of a binary outcome. It is highly interpretable (a "glass box" model) as it provides clear coefficients for each feature.
* **`C`** `[0.01, 0.1, 1, 10]`: The inverse of regularization strength. Smaller values specify stronger regularization to prevent overfitting by penalizing large coefficients.
* **`penalty`**`['l1', 'l2']`: The regularization method. L1 (Lasso) can perform feature selection by shrinking some coefficients to zero, L2 (Ridge) prevents overfitting without removing features.

### 3. Support Vector Machine (SVM)
The SVM seeks a hyperplane that maximizes the margin between classes. It is effective for high-dimensional data.
* **`C`** `[0.5, 0.8, 1, 2, 5, 10, 20]`: The regularization parameter. A smaller $C$ encourages a larger margin (preventing overfitting).
* **`kernel`** `['linear','rbf']`: Determines the decision boundary shape. We test a Linear kernel for separable data and RBF (Radial Basis Function) for non-linear relationships.

### 4. Decision Tree Classifier
A non-parametric model that learns simple decision rules from data features. It provides high interpretability.
* **`max_depth`** `[5, 10, 20]`: Controls the maximum depth of the tree to prevent overfitting (acting as regularization). 
* **`min_samples_split`** `[2, 10, 20]`: The minimum samples required to split an internal node. Higher values constrain the model from learning overly specific rules.
* **`criterion`** `['gini', 'entropy']`: The function to measure the quality of a split. We compare **Gini Impurity** against **Information Gain (Entropy)**.

### 5. Random Forest Classifier
An ensemble method that builds multiple decision trees and merges them to get a more accurate and stable prediction.
* **`n_estimators`** `[50, 70, 90, 110, 130]`: The number of trees in the forest. More trees generally improve stability at the cost of computation.
* **`max_depth`** `[8, 10, 12, 14]`: Limits the depth of individual trees to control model complexity.
* **`min_samples_leaf`** `[3, 4, 5, 6]`: The minimum number of samples required to be at a leaf node. Higher values smooth the model and reduce variance.
* **`min_samples_split`** `[2, 5, 10]`: Added to further control tree growth by specifying the minimum number of samples required to split an internal node.



In [5]:
models_config = [
        {
        'name': 'K-Nearest Neighbors (KNN)',
        'model': KNeighborsClassifier(),
        'params': {
            'classifier__n_neighbors': [3, 5, 7, 11, 15],
            'classifier__weights': ['uniform', 'distance'],
            'classifier__metric': ['euclidean', 'manhattan']
        }
        },
        {
        'name': 'Logistic Regression',
        'model': LogisticRegression(max_iter=10000, class_weight='balanced', random_state=42),
        'params': {
            'classifier__penalty': ['l1', 'l2'],
            'classifier__C': [0.01, 0.1, 1, 10],
            }
        },
        {
            'name': 'Support Vector Machine (SVM)',
            'model': SVC(probability=True, random_state=42, class_weight='balanced'),
            'params': {
                'classifier__C': [0.1, 1, 10],
                'classifier__kernel': ['linear', 'rbf']
            }
        },
        {
            'name': 'Decision Tree',
            'model': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
            'params': {
                'classifier__max_depth': [5, 10, 20],
                'classifier__min_samples_split': [2, 10, 20],
                'classifier__criterion': ['gini', 'entropy']
            }
        },
        {
            'name': 'Random Forest',
            'model': RandomForestClassifier(random_state=42, class_weight='balanced'),
            'params': {
                'classifier__n_estimators': [50, 70, 90, 110, 130],
                'classifier__max_depth': [8, 10, 12, 14],
                'classifier__min_samples_leaf': [3, 4, 5, 6],
                'classifier__min_samples_split': [2, 5, 10]
            }
        },
    ]


## Model Training and Evaluation Loop

This code iterates through the defined `models_config` to automate the training and tuning process. For each model:

1.  **Pipeline Integration**: The specific classifier is appended to the existing preprocessing steps (`pre_pipeline`).
2.  **Grid Search**: We employ `GridSearchCV` with **5-fold cross-validation** to find the optimal hyperparameters. The search optimizes for the **F1-score**, ensuring a balance between precision and recall, which better than **Accuracy** for our imbalanced dataset.
3.  **Testing & Metrics**: The best estimator found is immediately evaluated on the hold-out test set (`X_test`). We capture **Accuracy, Recall, Precision, and F1-Score** to generate a comprehensive performance comparison.

In [7]:
results_data = []


for config in models_config:
    print(f"--- TRAINING: {config['name']} ---")

    full_pipeline = Pipeline(steps=pre_pipeline.steps + [
        ('classifier', config['model'])
    ])
    grid_search = GridSearchCV(
        estimator=full_pipeline,
        param_grid=config['params'],
        cv=5,              
        scoring='f1', 
        n_jobs=-1,          
        verbose=1
    )

    grid_search.fit(X_train, y_train)
    

    best_model = grid_search.best_estimator_
    

    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1] 
    

    acc = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    

    results_data.append({
        'Model': config['name'],
        'Best Params': grid_search.best_params_,
        'Accuracy': acc,
        'Recall' : recall,
        'Precision': precision,
        'F1-Score': f1
    })

    print(f"Best params: {grid_search.best_params_}")
    print(f"F1 on test set: {f1}\n")



    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Stay', 'Exit'])
    disp.plot(cmap='Blues', values_format='d', ax=plt.gca())
    plt.title(f"Confusion Matrix: {config['name']}")
    filename = f"cm_{config['name']}.png"
    plt.savefig(filename) 
    plt.close() 


--- TRAINING: K-Nearest Neighbors (KNN) ---
Fitting 5 folds for each of 20 candidates, totalling 100 fits


Best params: {'classifier__metric': 'manhattan', 'classifier__n_neighbors': 3, 'classifier__weights': 'distance'}
F1 on test set: 0.47398843930635837

--- TRAINING: Logistic Regression ---
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best params: {'classifier__C': 0.01, 'classifier__penalty': 'l2'}
F1 on test set: 0.502092050209205

--- TRAINING: Support Vector Machine (SVM) ---
Fitting 5 folds for each of 6 candidates, totalling 30 fits


/Users/maksymiliankulicki/Documents/GitHub/banking_churn_pred/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/maksymiliankulicki/Documents/GitHub/banking_churn_pred/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/maksymiliankulicki/Documents/GitHub/banking_churn_pred/.venv/lib/python3.14/site-packages/s

Best params: {'classifier__C': 1, 'classifier__kernel': 'rbf'}
F1 on test set: 0.5920925747348119

--- TRAINING: Decision Tree ---
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best params: {'classifier__criterion': 'entropy', 'classifier__max_depth': 5, 'classifier__min_samples_split': 10}
F1 on test set: 0.5711805555555556

--- TRAINING: Random Forest ---
Fitting 5 folds for each of 240 candidates, totalling 1200 fits


/Users/maksymiliankulicki/Documents/GitHub/banking_churn_pred/.venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best params: {'classifier__max_depth': 14, 'classifier__min_samples_leaf': 5, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 130}
F1 on test set: 0.6326034063260341



In [9]:
results_df = pd.DataFrame(results_data).sort_values(by='F1-Score', ascending=False)
results_df.to_csv("model_results.csv", index=False)
print("Results saved in: model_results.csv")

print("="*60)
print("--- MODEL PERFORMANCE SUMMARY ---")
print("="*60)

print(results_df[['Model', 'Accuracy', 'F1-Score', 'Precision', 'Recall']])

Results saved in: model_results.csv
--- MODEL PERFORMANCE SUMMARY ---
                          Model  Accuracy  F1-Score  Precision    Recall
4                 Random Forest    0.8490  0.632603   0.628019  0.637255
2  Support Vector Machine (SVM)    0.7885  0.592093   0.488076  0.752451
3                 Decision Tree    0.7530  0.571181   0.442204  0.806373
1           Logistic Regression    0.7025  0.502092   0.381194  0.735294
0     K-Nearest Neighbors (KNN)    0.8180  0.473988   0.577465  0.401961
